### Silver: streaming Vehicle Positions
##### Bronze stores each vehicle entity as a raw_json blob (see bronze_gtfs_vehicle_positions), e.g. `{"trip": {"trip_id": ..., "route_id": "APS_2f"}, "position": {"latitude": -33.95, "longitude": 150.78}, "timestamp": "1790030848", ...}`.
##### route_id, vehicle.label and occupancy_status are legitimately null on a chunk of rows (unscheduled trips, older rolling stock) -- sampled from Bronze at ~7%, ~12%, ~85% missing respectively. Nullable by design, not bad data.
##### Writes an append-only typed history table plus a quarantine table for anything from_json can't parse or that's missing a usable id/coordinates.

In [0]:
%run ../../commonsetup/Setup-Common

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
import sys, os
sys.path.append(os.path.abspath("../.."))
from utils.silver_transforms import upsert_current_position

### Config
##### Reads the single bronze_gtfs_vehicle_positions table (both modes, split by gtfs_mode) -- no per-mode read needed here.

In [0]:
ENTITY                   = "vehicle_positions"
 
BRONZE_TABLE            = f"{CATALOG_NAME}.{SCHEMA_NAME}.bronze_gtfs_{ENTITY}"
SILVER_TABLE            = f"{CATALOG_NAME}.{SCHEMA_NAME}.silver_gtfs_{ENTITY}"
QUARANTINE_TABLE        = SILVER_TABLE + "_quarantine"
CHECKPOINT_HISTORY      = f"{CHECKPOINT_BASE}/silver_gtfs_{ENTITY}/checkpoint"
CHECKPOINT_QUARANTINE   = f"{CHECKPOINT_BASE}/silver_gtfs_{ENTITY}_quarantine/checkpoint"

In [0]:
# ---- Schema of the JSON stored in raw_json ------------------------------
raw_json_schema = StructType([
    StructField("trip", StructType([
        StructField("trip_id", StringType()),
        StructField("route_id", StringType()),
        StructField("schedule_relationship", StringType()),
    ])),
    StructField("vehicle", StructType([
        StructField("id", StringType()),
        StructField("label", StringType()),
    ])),
    StructField("position", StructType([
        StructField("latitude", DoubleType()),
        StructField("longitude", DoubleType()),
    ])),
    StructField("timestamp", StringType()),
    StructField("stop_id", StringType()),
    StructField("congestion_level", StringType()),
    StructField("occupancy_status", StringType()),
])



In [0]:
# ---- Read Bronze as a stream ---------------------------------------------
bronze_df = spark.readStream.table(BRONZE_TABLE)
 
parsed_df = (
    bronze_df
    .withColumn("parsed", F.from_json(F.col("raw_json"), raw_json_schema))
    .select(
        F.col("entity_id"),
        F.col("gtfs_mode"),
        F.col("poll_timestamp").cast("timestamp").alias("poll_timestamp"),
        F.col("ingestion_timestamp").cast("timestamp").alias("ingestion_timestamp"),
        F.col("parsed.vehicle.id").alias("vehicle_id"),
        F.col("parsed.vehicle.label").alias("vehicle_label"),
        F.col("parsed.trip.trip_id").alias("trip_id"),
        F.col("parsed.trip.route_id").alias("route_id"),
        F.col("parsed.trip.schedule_relationship").alias("schedule_relationship"),
        F.col("parsed.position.latitude").alias("latitude"),
        F.col("parsed.position.longitude").alias("longitude"),
        # GTFS-RT timestamp is epoch seconds, delivered here as a string.
        F.to_timestamp(F.col("parsed.timestamp").cast("long")).alias("position_timestamp"),
        F.col("parsed.stop_id").alias("stop_id"),
        F.col("parsed.congestion_level").alias("congestion_level"),
        F.col("parsed.occupancy_status").alias("occupancy_status"),
        F.col("raw_json").alias("_raw_json"),   # kept only for the quarantine branch
    )
)

In [0]:
# Missing vehicle_id/coordinates -> quarantine with the original JSON, not a silent drop.
good_df = parsed_df.filter(
    F.col("vehicle_id").isNotNull()
    & F.col("latitude").isNotNull()
    & F.col("longitude").isNotNull()
)
bad_df = parsed_df.filter(
    F.col("vehicle_id").isNull()
    | F.col("latitude").isNull()
    | F.col("longitude").isNull()
)

### Current-position table
##### History answers "where has this vehicle been"; a live map wants "where is it right now" -- one row per vehicle_id, kept current via foreachBatch + MERGE (upsert_current_position in utils/silver_transforms.py), rather than a separate scheduled row_number() batch job.

In [0]:
# ---- Write: append-only Silver history ------------------------------------
history_query = (
    good_df.drop("_raw_json").writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_HISTORY)
    .partitionBy("gtfs_mode")
    .trigger(availableNow=True)          # swap to .trigger(processingTime="30 seconds") for a continuously running job
    .toTable(SILVER_TABLE)
)

# ---- Write: quarantine ------------------------------------------------------
quarantine_query = (
    bad_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_QUARANTINE)
    .trigger(availableNow=True)
    .toTable(QUARANTINE_TABLE)
)

history_query.awaitTermination()
quarantine_query.awaitTermination()

CURRENT_TABLE      = f"{CATALOG_NAME}.{SCHEMA_NAME}.silver_gtfs_vehicle_positions_current"
CHECKPOINT_CURRENT = f"{CHECKPOINT_BASE}/silver_gtfs_{ENTITY}_current/checkpoint"

current_query = (
    good_df.drop("_raw_json").writeStream
    .foreachBatch(lambda batch_df, batch_id: upsert_current_position(spark, batch_df, CURRENT_TABLE))
    .option("checkpointLocation", CHECKPOINT_CURRENT)
    .trigger(availableNow=True)
    .start()
)
current_query.awaitTermination()

In [0]:
%sql
select * from silver_gtfs_trips